# 离线预览：人工答案，不是 Jev 实测

# TypeSafe 应用场景实验（Use Case Map Lab）

针对官方文档对应章节的可运行实验笔记，全部实验使用**中文场景与中文提示词**。
面向会基础 Python、刚接触 AI Agent 的读者。

**学习目标：** 按输入、问题、输出、动作、验证定义新任务，并实现候选相关性与引用支撑两个小配方。

[官方原文](https://docs.typesafe.ai/concepts/use-case-map) · [中文参考](https://bald0wang.github.io/jev-docs-zh/concepts/use-case-map/)。本章以中文重述理论、复刻对应场景；扩展实验会单独说明。
所有客户、订单及消息均为教学合成数据。

## 笔记本结构

| 章节 | 内容 |
|---|---|
| 0. 准备 | 安装库、配置客户端、连通性测试与离线示例 |
| 1 | 应用类别与决策形态 |
| 2 | 搜索与检索：候选相关性 |
| 3 | 科研核验：证据是否支持论断 |
| 4 | 自己的任务卡 |
| 练习与小结 | 练习、自查、总结与本次执行记录 |

实验按**原理 → 理论根基 → 定义数据 → 定义问题 → 调用 → 解读结果**展开，每个代码单元格只做一件事。

## 运行要求

- Python ≥ 3.10；本章使用 `typesafe-sdk==0.7.0`。
- 真实实验需要启动进程的 `TYPESAFE_API_KEY` 环境变量，密钥不要写进 Notebook。

在本仓库 `notebooks/` 目录创建环境并打开本文件：

```bash
./setup_env.sh
.venv/bin/python -m pip install -r requirements.txt -c generators/constraints-foundations.txt
.venv/bin/jupyter lab use_case_map_experiments.ipynb
```

产品名、字段名和选项 key 保持英文，state、提示词与解说使用中文。
默认 `JEV_RUN_MODE=live`，调用失败即停止；无密钥学习时，在启动 Jupyter 前设置 `JEV_RUN_MODE=offline`。
`auto` 仅供教学体验，缺密钥或 401 时显式回退；正式验收使用 `live`。

**验证状态：真实 API 待验收。** 本文件尚未执行真实 API；离线检查仅验证代码路径。
批量执行、离线预览和验收记录见本目录 `MAINTENANCE.md`。

## 0. 准备

本节可折叠阅读，但独立运行时不能跳过。客户端、辅助对象和示例数据都在本文件中定义。

### 0.1 安装依赖

推荐先运行 `setup_env.sh`。只有当前内核缺少 SDK 时，本格才安装依赖。

In [1]:
import importlib.util
if importlib.util.find_spec("typesafe_sdk") is None:
    %pip install -q typesafe-sdk==0.7.0

**观察与理解：** 安装包的名字是 typesafe-sdk，Python 导入名是 typesafe_sdk。安装成功不代表 API 已连通。

### 0.2 导入与配置

默认模型固定版本，便于记录实验条件；可通过环境变量更换。不要从 Notebook 输入密钥。

In [2]:
import os
import json
import time
import math
from datetime import datetime, timezone
from importlib.metadata import version
from typesafe_sdk import (
    Choice, Score, Noul, NoulCriteria, TypeSafeClient,
    TypeSafeAuthenticationError, RetryPolicy,
)

MODEL = os.environ.get("TYPESAFE_DEFAULT_MODEL", "jev-1.13.0")
RUN_MODE = os.environ.get("JEV_RUN_MODE", "live")
API_KEY = os.environ.get("TYPESAFE_API_KEY", "")
if RUN_MODE not in {"live", "offline", "auto"}:
    raise ValueError("JEV_RUN_MODE 只能是 live、offline 或 auto")
if RUN_MODE == "live" and not API_KEY:
    raise RuntimeError("请在启动 Jupyter 前配置 TYPESAFE_API_KEY 环境变量")
client = None
if RUN_MODE != "offline" and API_KEY:
    client = TypeSafeClient(api_key=API_KEY, model=MODEL, timeout=30,
                           retry=RetryPolicy(max_retries=0))
print("模式：", RUN_MODE, "SDK：", version("typesafe-sdk"), "模型配置：", MODEL)

模式： offline SDK： 0.7.0 模型配置： jev-1.13.0


正式验收禁用自动回退，且不自动重试，以便请求数量有界。`auto` 与 `offline` 是教学工具，不代表成功连接模型。

### 0.3 连通性测试

用一条 Noul 检查真实响应能否返回。网络、限流与输入错误直接抛出，不伪装成不确定判断。

In [3]:
PING = {"source": "offline", "reason": "未发起连通性请求"}
if client is not None:
    try:
        ping = client.system_one("你好", {"greeting": Noul(
            instructions="这段文字是否在打招呼？")})
        PING = {"source": "live", "model": ping.model,
                "input_tokens": ping.usage.input_tokens,
                "output_tokens": ping.usage.output_tokens}
    except TypeSafeAuthenticationError:
        if RUN_MODE == "live":
            raise
        client.close()
        client = None
        PING["reason"] = "401 鉴权失败，仅教学模式允许回退"
print(json.dumps(PING, ensure_ascii=False))

{"source": "offline", "reason": "未发起连通性请求"}


**观察与理解：** source=live 表示这一次连通性请求成功；仍要查看后续实验记录，不能用它代替整章验收。

### 0.4 离线替身

沿用参考模板的 `_FakeAnswer` 与 `_FakeResponse` 访问方式。人工数字仅用来检验读取字段和代码分支。

In [4]:
class _FakeAnswer:
    def __init__(self, type_, **values):
        self.type = type_
        for name, value in values.items():
            setattr(self, name, value)


class _FakeResponse:
    def __init__(self, answers):
        self.answers = answers
        self.choices = {k: v for k, v in answers.items() if v.type == "choice"}
        self.scores = {k: v for k, v in answers.items() if v.type == "score"}
        self.nouls = {k: v for k, v in answers.items() if v.type == "noul"}
        self.model = "人工示例，非模型预测"
        self.usage = _FakeAnswer("usage", input_tokens=0, output_tokens=0)

人工 Score 由概率计算期望，避免模板中的分数与分布不一致。人工 confidence 只是指定的演示字段，不是在复现服务端的计算公式。

定义两种示例答案构造器；Noul 可直接用 `_FakeAnswer`。所有具体答案集中在下一节。

In [5]:
def fake_choice(probabilities, confidence):
    return _FakeAnswer("choice", choice=max(probabilities, key=probabilities.get),
                       probabilities=probabilities, confidence=confidence)


def fake_score(probabilities, legend, confidence):
    return _FakeAnswer("score", score=sum(k * p for k, p in probabilities.items()),
                       probabilities=probabilities, legend=dict(enumerate(legend)),
                       confidence=confidence)

**观察与理解：** 例如概率 {0:0.05, 1:0.26, 2:0.69} 对应 1.64。不能把另一个数与该分布配在一起。

### 0.5 统一调用入口

每次调用记录来源、模型与 token 用量。离线耗时记为 None，不把本地字典访问当成模型速度。

In [6]:
CALL_LOG = []


class TS:
    def call(self, state, questions, offline_answers, label):
        start = time.perf_counter()
        source = "live"
        if client is None:
            response, source = _FakeResponse(offline_answers), "offline"
        else:
            try:
                response = client.system_one(state, questions)
            except TypeSafeAuthenticationError:
                if RUN_MODE != "auto":
                    raise
                response, source = _FakeResponse(offline_answers), "offline"
        validate_response(response, questions)
        CALL_LOG.append({"case": label, "source": source, "model": response.model,
                         "seconds": time.perf_counter() - start if source == "live" else None,
                         "input_tokens": response.usage.input_tokens,
                         "output_tokens": response.usage.output_tokens})
        if source == "offline":
            print("离线示例：", label, "；人工答案，不是 Jev 实测")
        return response


ts = TS()

与参考模板相比，这里增加了严格 live 模式和逐次记录。保留 401 教学回退，但超时、429 等错误继续失败，防止验收被回退掩盖。

校验结构与数值契约；只断言接口应满足的性质，不断言真实模型必须预测某个标签。

In [7]:
def validate_response(response, questions):
    if set(response.answers) != set(questions):
        raise ValueError("答案 ID 与问题 ID 不一致")
    for key, question in questions.items():
        answer = response.answers[key]
        if isinstance(question, Noul):
            if not 0 <= answer.noul <= 1:
                raise ValueError("Noul 超出概率范围")
            continue
        probabilities = answer.probabilities
        if not all(math.isfinite(p) and 0 <= p <= 1 for p in probabilities.values()):
            raise ValueError("概率值无效")
        if not math.isclose(sum(probabilities.values()), 1, abs_tol=0.02):
            raise ValueError("概率之和偏离 1")
        if not 0 <= answer.confidence <= 1:
            raise ValueError("confidence 超出范围")
        if isinstance(question, Choice):
            if set(probabilities) != set(question.criteria):
                raise ValueError("Choice 选项集合不一致")
            if answer.choice not in probabilities:
                raise ValueError("Choice 标签不在选项中")
        else:
            expected = sum(int(k) * p for k, p in probabilities.items())
            if not math.isclose(answer.score, expected, abs_tol=0.03):
                raise ValueError("Score 与概率加权期望不一致")

**观察与理解：** 容差用于服务端数值舍入。结构检查通过只说明响应可读取，不证明语义判断正确。

显示结果时统一列出类型、概率和置信度；Noul 不额外制造 confidence 字段。

In [8]:
def show(response):
    rows = {}
    for key, answer in response.answers.items():
        rows[key] = {name: getattr(answer, name) for name in
                     ("type", "choice", "score", "noul", "confidence", "probabilities", "legend")
                     if hasattr(answer, name)}
    print(json.dumps(rows, ensure_ascii=False, indent=2))

### 0.6 本章离线示例数据

以下数值全部人工构造，专门测试分支；不来自 Jev，也不能用于估计中文准确率或校准情况。正式 live 运行不会使用这些答案。

In [9]:
RELEVANCE_OFFLINE = {
    "candidate_0": fake_score({0: 0.02, 1: 0.03, 2: 0.95},
        ["无关", "相关但不能直接回答", "直接回答查询"], 0.93),
    "candidate_1": fake_score({0: 0.97, 1: 0.02, 2: 0.01},
        ["无关", "相关但不能直接回答", "直接回答查询"], 0.95),
    "candidate_2": fake_score({0: 0.08, 1: 0.85, 2: 0.07},
        ["无关", "相关但不能直接回答", "直接回答查询"], 0.82),
}
EVIDENCE_OFFLINE = {
    "claim_0_supported": _FakeAnswer("noul", noul=0.96),
    "claim_1_supported": _FakeAnswer("noul", noul=0.04),
}

## 1. 📖 理论根基：用输入与动作约束想法

用例地图提供头脑风暴方向，不表示模型已在这些行业通过业务验收。
自动化软件、实时应用、大数据 Map Reduce、通用验证与 Harness 工程，是文档列出的五类应用形态。
共同点是把语言理解接入代码；是否满足吞吐、成本、时延和质量要求，仍要测量。

| 决策形态 | 问题与输出 | 后续动作 | 最小验证 |
|---|---|---|---|
| 分类 | Choice 类别 | 分类存储 | 混淆矩阵、other 比例 |
| 检测 | Noul 属性概率 | 标记或复核 | 漏检、误报与阈值 |
| 评分 | Score 有序等级 | 排优先级 | 与人工量表对照 |
| 路由 | 分类与不确定性 | 选择代码路径 | 覆盖率、错误路径 |
| 搜索 | 查询与候选的关系 | 选出匹配候选 | 候选集召回与命中 |
| 检索 | 上下文相关性 | 送入下一阶段 | 证据覆盖、遗漏 |
| 排序 | 相关性分数或比较 | 对候选排序 | 人工排序对照 |
| 核验 | 具体命题是否成立 | 接受、修改、复核 | 错误类型检出情况 |
| ML 特征提取 | 语义属性概率 | 输入下游模型 | 独立标签上的增益 |
| 结构化提取 | 有限候选的字段判断 | 填入业务记录 | 字段准确率、缺失率 |

TypeSafe 的有限答案原语不等同于任意文本抽取器。未知姓名、日期等开放字段通常还需要候选生成或提取流程。

### 从行业回到具体判断

| 行业或场景 | 可以先定义的一项判断 |
|---|---|
| 科学发现 | 段落是否满足综述纳入标准 |
| 模型路由 | 请求属于哪个领域、难度等级 |
| LLM 防护栏 | 输入或工具调用是否违反明确规则 |
| 语义代码检查 | 变更是否符合一条写作或代码规范 |
| 招聘 | 材料是否提供岗位所需经验的证据 |
| 销售线索 | 公司档案是否匹配客户画像 |
| 客户支持 | 主诉属于哪个支持队列 |
| 保险理赔 | 材料是否缺少指定信息 |
| 金融犯罪调查 | 描述中是否存在指定的可疑信号 |
| 法律与合规 | 材料是否包含指定条款 |
| 电商市场 | 商品属于哪个预定义类别 |
| 内容审核 | 是否违反给定社区政策 |
| 广告 | 文案与落地页信息是否一致 |
| 游戏 | 玩家举报属于哪类问题 |
| 风险评估 | 报告中某个风险信号是否存在 |
| 需求预测 | 文本是否表达购买意图 |
| 图与知识图谱 | 给定两条记录是否相互矛盾 |

每项都需要相应上下文和人工标准。下面只实现“搜索与检索”和“科学发现中的引用核验”两个低依赖示例。

## 2. 配方 A：为候选段落打相关性分

### 原理与理论根基

对应原文搜索与检索条目：先有候选集，再评价查询与候选之间的关系。本例不实现全库索引或完整 RAG。
三个问题共享一个 state，每个问题明确引用自己的候选。


[官方原文](https://docs.typesafe.ai/concepts/use-case-map) · [中文参考](https://bald0wang.github.io/jev-docs-zh/concepts/use-case-map/) · [官方原文](https://docs.typesafe.ai/primitives/score) · [中文参考](https://bald0wang.github.io/jev-docs-zh/primitives/score/)

### 第一步：准备查询与候选

In [10]:
SEARCH_STATE = {
    "query": "订单被重复扣款应该怎么处理？",
    "candidates": [
        {"id": "D1", "text": "如同一订单被重复扣款，请提交订单编号与扣款凭证，客服核查后可退还重复部分。"},
        {"id": "D2", "text": "更改登录密码请进入账户设置，选择安全与密码。"},
        {"id": "D3", "text": "账单可在每月结算后下载，页面列出每笔扣款金额。"},
    ],
}

### 第二步：明确‘相关’的等级

In [11]:
RELEVANCE_LEVELS = ["无关", "相关但不能直接回答", "直接回答查询"]
RELEVANCE_QUESTIONS = {
    f"candidate_{i}": Score(
        instructions=f"`candidates[{i}].text` 与 `query` 有多相关？只评价这一个候选。",
        criteria=RELEVANCE_LEVELS,
    )
    for i in range(len(SEARCH_STATE["candidates"]))
}

**观察与理解：** ‘提到扣款’与‘回答重复扣款怎么办’的相关程度不同，量表要体现这一区别。

### 第三步：一次请求所有候选判断

In [12]:
search_response = ts.call(SEARCH_STATE, RELEVANCE_QUESTIONS, RELEVANCE_OFFLINE, "候选相关性")

离线示例： 候选相关性 ；人工答案，不是 Jev 实测


### 第四步：查看原始分数与分布

In [13]:
show(search_response)

{
  "candidate_0": {
    "type": "score",
    "score": 1.93,
    "confidence": 0.93,
    "probabilities": {
      "0": 0.02,
      "1": 0.03,
      "2": 0.95
    },
    "legend": {
      "0": "无关",
      "1": "相关但不能直接回答",
      "2": "直接回答查询"
    }
  },
  "candidate_1": {
    "type": "score",
    "score": 0.04,
    "confidence": 0.95,
    "probabilities": {
      "0": 0.97,
      "1": 0.02,
      "2": 0.01
    },
    "legend": {
      "0": "无关",
      "1": "相关但不能直接回答",
      "2": "直接回答查询"
    }
  },
  "candidate_2": {
    "type": "score",
    "score": 0.99,
    "confidence": 0.82,
    "probabilities": {
      "0": 0.08,
      "1": 0.85,
      "2": 0.07
    },
    "legend": {
      "0": "无关",
      "1": "相关但不能直接回答",
      "2": "直接回答查询"
    }
  }
}


**观察与理解：** Score 不是概率。归一化后的 Score 也不是‘答案正确概率’。

### 第五步：排序并保留‘没有合适候选’出口

In [14]:
ranked = sorted(
    [{"id": item["id"], "score": search_response.scores[f"candidate_{i}"].score,
      "confidence": search_response.scores[f"candidate_{i}"].confidence}
     for i, item in enumerate(SEARCH_STATE["candidates"])],
    key=lambda row: row["score"], reverse=True,
)
selected = [row for row in ranked if row["score"] >= 1.5 and row["confidence"] >= 0.75]

显示排序与筛选结果。

In [15]:
print(json.dumps({"排序": ranked, "送往下一阶段": selected or "无合适候选，继续检索或复核"},
                 ensure_ascii=False, indent=2))

{
  "排序": [
    {
      "id": "D1",
      "score": 1.93,
      "confidence": 0.93
    },
    {
      "id": "D3",
      "score": 0.99,
      "confidence": 0.82
    },
    {
      "id": "D2",
      "score": 0.04,
      "confidence": 0.95
    }
  ],
  "送往下一阶段": [
    {
      "id": "D1",
      "score": 1.93,
      "confidence": 0.93
    }
  ]
}


**观察与理解：** 阈值只是教学值；即使 top-1 排得出来，也可能整个候选集都无关。评估排序前还应检查候选集是否覆盖答案。

## 3. 配方 B：证据是否支持论断

### 原理与理论根基

对应原文科学发现与通用验证条目。给定证据和候选论断，逐项问证据是否足以支持。
本例使用一项虚构课堂试验，不涉及真实研究结论。没有支持不一定意味着论断本身为假。

准备证据与两个论断。

In [16]:
EVIDENCE_STATE = {
    "evidence": "课堂小组在同一台电脑上做了十次测试。方法甲的平均耗时为 12 秒，方法乙为 9 秒。未测试其他设备。",
    "claims": [
        "这十次测试中，方法乙的平均耗时比方法甲短。",
        "方法乙在所有设备和所有任务上都优于方法甲。",
    ],
}

问题要求只按所给证据判断，避免外部常识补造支撑。

In [17]:
EVIDENCE_QUESTIONS = {
    f"claim_{i}_supported": Noul(instructions=(
        f"仅根据 `evidence`，它是否充分支持 `claims[{i}]` 的完整论断？不要用外部知识补充证据。"))
    for i in range(len(EVIDENCE_STATE["claims"]))
}

**观察与理解：** 第二条增加了‘所有设备和所有任务’的范围，重点观察模型是否注意到证据的适用范围。

调用。

In [18]:
evidence_response = ts.call(EVIDENCE_STATE, EVIDENCE_QUESTIONS, EVIDENCE_OFFLINE, "引用支撑")

离线示例： 引用支撑 ；人工答案，不是 Jev 实测


逐条查看概率。

In [19]:
for i, claim in enumerate(EVIDENCE_STATE["claims"]):
    print({"论断": claim, "给定证据支持的概率": evidence_response.nouls[f"claim_{i}_supported"].noul})

{'论断': '这十次测试中，方法乙的平均耗时比方法甲短。', '给定证据支持的概率': 0.96}
{'论断': '方法乙在所有设备和所有任务上都优于方法甲。', '给定证据支持的概率': 0.04}


**观察与理解：** 两条合成输入只能检查示例行为。真实引用核验还需包含部分支持、矛盾、遗漏和不同表述方式的独立样本。

## 4. 用任务卡定义你的下一个配方

| 字段 | 本章相关性配方 | 你需要填写的内容 |
|---|---|---|
| 输入 | 查询与候选段落 | 只列判断所需事实 |
| 问题 | 候选是否直接回答查询 | 一个明确属性 |
| 输出 | 0–2 的 Score 与分布 | 答案空间与量表 |
| 动作 | 排序、筛选或继续检索 | 由代码实施的动作 |
| 验证 | 人工相关性、排序与遗漏 | 样本、标签、错误成本 |
| 不确定出口 | 无合适候选 | 复核、补充信息或停止 |

原先的学习社区工单与 FAQ 教程保留在项目 `docs/` 和 `examples/`，作为迁移练习；它们的人工响应不计入本系列真实结果。

## 练习与自查

从行业表选一个任务，填写任务卡；给出一个明确正例、一个明确反例和一个材料不足的例子。说明哪种错误代价更高。

<details><summary>参考思路：先完成练习再展开</summary>

例如课程答疑：输入问题与 FAQ 候选，问题为候选能否直接回答，输出相关性 Score，动作是显示候选或交助教；漏掉合适答案与展示错误答案要分别统计。材料不足时允许空结果。

</details>

## 小结

| 小配方 | 模型做什么 | 程序做什么 |
|---|---|---|
| 相关性 | 按量表判断 | 排序、筛选与空结果处理 |
| 引用支撑 | 判断论断与证据关系 | 记录、复核或请求修改 |

延伸阅读：[官方原文](https://docs.typesafe.ai/cookbooks/citation_check) · [中文参考](https://bald0wang.github.io/jev-docs-zh/cookbooks/citation_check/) · [官方原文](https://docs.typesafe.ai/cookbooks/rerank_typesafe) · [中文参考](https://bald0wang.github.io/jev-docs-zh/cookbooks/rerank_typesafe/)

离线运行只说明教材代码能执行。正式交付必须实际运行 live，并阅读每条输出；缺失的分支应记为未观察到。

## 本次执行记录

先关闭连接，再生成记录。下面的 JSON 由实际运行计算，批量执行器会据此检查来源。

In [20]:
if client is not None:
    client.close()

真实探针只演示行为路径；若据其返回挑选样例，这批样例就不适合再当作无偏准确率测试集。延迟也只是本次网络环境中的观测。

In [21]:
AUDIT = {
    "kind": "jev_execution_audit",
    "executed_at_utc": datetime.now(timezone.utc).isoformat(),
    "sdk": version("typesafe-sdk"), "requested_model": MODEL,
    "mode": RUN_MODE, "ping": PING,
    "real_calls": sum(x["source"] == "live" for x in CALL_LOG),
    "offline_calls": sum(x["source"] == "offline" for x in CALL_LOG),
    "cases": CALL_LOG,
    "coverage": globals().get("COVERAGE", {}),
    "validation_status": "live_executed_requires_review" if (
        PING["source"] == "live" and CALL_LOG
        and all(x["source"] == "live" for x in CALL_LOG)
    ) else "offline_only_not_model_evidence",
}
print(json.dumps(AUDIT, ensure_ascii=False, indent=2))

{
  "kind": "jev_execution_audit",
  "executed_at_utc": "2026-09-23T15:36:37.682683+00:00",
  "sdk": "0.7.0",
  "requested_model": "jev-1.13.0",
  "mode": "offline",
  "ping": {
    "source": "offline",
    "reason": "未发起连通性请求"
  },
  "real_calls": 0,
  "offline_calls": 2,
  "cases": [
    {
      "case": "候选相关性",
      "source": "offline",
      "model": "人工示例，非模型预测",
      "seconds": null,
      "input_tokens": 0,
      "output_tokens": 0
    },
    {
      "case": "引用支撑",
      "source": "offline",
      "model": "人工示例，非模型预测",
      "seconds": null,
      "input_tokens": 0,
      "output_tokens": 0
    }
  ],
  "coverage": {},
  "validation_status": "offline_only_not_model_evidence"
}


读完输出后，在本仓库 `notebooks/MAINTENANCE.md` 的验收表中记录日期、真实模型、观察到的分支和偏离预期之处。不要把人工演示数值抄进实测记录。